In [2]:
import torch.optim as optim
import numpy as np

# For visualization
from tqdm.notebook import tqdm

In [ ]:
def compute_miou(pred, truth, num_classes):
    ious = []
    pred = pred.flatten()
    truth = truth.flatten()
    for clas in range(num_classes):
        pred_clas = pred == clas
        truth_clas = truth == clas
        intersection = (pred_clas & truth_clas).sum().item()
        union = (pred_clas | truth_clas).sum().item()
        if union == 0:
            ious.append(float("nan"))
        else:
            ious.append(intersection/union)
    return np.nanmean(ious), ious

## Trainingsloop:
In jedem Schritt:
- Eingabebilder und zugehörige Labels auf `device` laden
- Vorwärtsberechnung `forward pass` durchführen
- Verlust `loss` berechnen
- Gradienten mit `loss.backward()` berechnen
- Parameter mit `optimizer.step()` aktualisieren

In [ ]:
def train(dataloader, model, loss_function, optimizer, device):
    # Switch to train mode
    model.train()

    # For storing running metrics
    epoch_loss, epoch_acc, epoch_miou = 0, 0, 0

    # Create a progress bar
    pbar = tqdm(dataloader, decs="training loop", leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        ### Forward pass
        # Get output/prediction
        output = model(images)
        # Calculate loss
        loss = loss_function(output, labels)

        ### Backpropagation
        # Gradients have to be zeroed for each batch
        optimizer.zero_grad()
        # Calculate the gradients
        loss.backward()
        # Update the parameters
        optimizer.step()

        ### Calculate metrics per batch
        prediction = output.argmax(dim=1)

        acc = (prediction == labels).float().mean()
        miou, _ = compute_miou(prediction, labels, model.num_classes)
        epoch_loss += loss.item()
        epoch_acc += acc.item()
        epoch_miou += miou

    num_batches = len(dataloader)
    # Return metrics averaged across all batches
    return epoch_loss/num_batches, epoch_acc/num_batches, epoch_miou/num_batches